# TextExplanationComponent Demo (20 Newsgroups)

This notebook demonstrates `TextExplanationComponent` from `humaine-explainerdashboard` using the standard `20 Newsgroups` text dataset from scikit-learn.

**Pipeline overview:**
1. Load the 20 Newsgroups dataset
2. Train a TF-IDF + Logistic Regression text classifier
3. Wrap model outputs in a `ClassifierExplainer`
4. Build a `predict_fn` closure for LIME
5. Mount `TextExplanationComponent` in an `ExplainerDashboard`


## 0 · Dependencies

Install any missing packages before running:
```bash
pip install humaine-explainerdashboard scikit-learn lime
```


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from explainerdashboard import ClassifierExplainer, ExplainerDashboard
from explainerdashboard.dashboard_components import TextExplanationComponent


## 1 · Load a standard text demo dataset


In [3]:
categories = [
    "sci.space",
    "rec.sport.hockey",
    "comp.graphics",
    "talk.politics.mideast",
]

train = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes"),
)
test = fetch_20newsgroups(
    subset="test",
    categories=categories,
    remove=("headers", "footers", "quotes"),
)

train_texts = pd.Series(train.data, name="text").fillna("")
test_texts = pd.Series(test.data, name="text").fillna("")
y_train = pd.Series(train.target, name="target")
y_test = pd.Series(test.target, name="target")
class_names = list(train.target_names)

print(f"Train rows: {len(train_texts):,} | Test rows: {len(test_texts):,}")
print(f"Classes ({len(class_names)}): {class_names}")


Train rows: 2,341 | Test rows: 1,558
Classes (4): ['comp.graphics', 'rec.sport.hockey', 'sci.space', 'talk.politics.mideast']


## 2 · Vectorize text and train the model


In [4]:
vectorizer = TfidfVectorizer(max_features=3000, stop_words="english")

X_train_sparse = vectorizer.fit_transform(train_texts)
X_test_sparse = vectorizer.transform(test_texts)

feature_names = vectorizer.get_feature_names_out()
X_train = pd.DataFrame(X_train_sparse.toarray(), columns=feature_names, index=train_texts.index)
X_test = pd.DataFrame(X_test_sparse.toarray(), columns=feature_names, index=test_texts.index)

model = LogisticRegression(max_iter=1000, multi_class="auto")
model.fit(X_train, y_train)

acc = model.score(X_test, y_test)
print(f"Test accuracy: {acc:.1%} | test samples: {len(X_test)}")


Test accuracy: 90.6% | test samples: 1558


## 3 · Build the ClassifierExplainer


In [5]:
explainer = ClassifierExplainer(
    model,
    X_test,
    y_test,
    labels=class_names,
    index_name="Doc ID",
)


Note: model_output='probability' is currently not supported for linear classifiers models with shap. So defaulting to model_output='logodds' If you really need probability outputs use shap='kernel' instead.
Note: shap values for shap='linear' get calculated against X_background, but paramater X_background=None, so using X instead...
Generating self.shap_explainer = shap.LinearExplainer(model, X)...


## 4 · Define the `predict_fn` closure for LIME

LIME perturbs the raw text and calls `predict_fn` with lists of perturbed strings.
This closure reproduces the same TF-IDF preprocessing used during training.


In [6]:
def predict_fn(texts: list[str]) -> np.ndarray:
    """Inference pipeline for LIME: text list -> class probability matrix."""
    X_sparse = vectorizer.transform(pd.Series(texts).fillna(""))
    X_df = pd.DataFrame(X_sparse.toarray(), columns=feature_names)
    return model.predict_proba(X_df)


## 5 · Create the TextExplanationComponent

`raw_texts` must be indexed exactly like the explainer dataset (`X_test.index`).


In [7]:
raw_texts_test = pd.DataFrame({"text": test_texts}, index=X_test.index)

text_component = TextExplanationComponent(
    explainer,
    predict_fn=predict_fn,
    raw_texts=raw_texts_test,
    text_col="text",
    class_names=class_names,
    num_features=15,
    title="20 Newsgroups Text Explanation",
    subtitle="Select a document and click Explain to view LIME word contributions",
)


## 6 · Launch the dashboard

Open the printed URL, choose a `Doc ID`, and click **Explain** to run LIME.

> Use `mode='external'` to keep the notebook interactive while the server runs.


In [8]:
db = ExplainerDashboard(
    explainer,
    [text_component],
    title="20 Newsgroups - Text Explanation Demo",
)
db.run(port=8050)


Building ExplainerDashboard..
Detected notebook environment, consider setting mode='external', mode='inline' or mode='jupyterlab' to keep the notebook interactive while the dashboard is running...
Generating layout...
Calculating dependencies...
Reminder: you can store the explainer (including calculated dependencies) with explainer.dump('explainer.joblib') and reload with e.g. ClassifierExplainer.from_file('explainer.joblib')
Registering callbacks...
Starting ExplainerDashboard on http://192.168.1.72:8050


In [ ]:
# Run this cell to stop the server
db.terminate(port=8050)
